# Test Prediction Request ke Model Serving
### Marketing Campaign Response Prediction — TF Serving

Notebook ini menguji model yang sudah di-serve oleh **TensorFlow Serving**
(lihat `Dockerfile`, dijalankan via `docker compose up --build` atau setelah
di-deploy ke Railway — lihat `DEPLOYMENT.md`).

Model menerima input dalam format `tf.Example` yang di-serialize sebagai
base64 string di dalam JSON request (format standar REST API TF Serving
untuk signature yang menerima `serialized_tf_examples`).

**Sebelum menjalankan notebook ini**, pastikan container TF Serving sudah
jalan, misalnya:
```bash
PORT=8501 docker compose up --build
```
atau kalau menguji app yang sudah di-deploy ke Heroku, ganti `BASE_URL` di
bawah dengan URL Heroku app kamu.

In [1]:
import requests
import json
import base64
import tensorflow as tf
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

I0000 00:00:1788167634.015695   36160 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788167634.071844   36160 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/conda/envs/tfx-env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
I0000 00:00:1788167635.864475   36160 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
BASE_URL = BASE_URL = "http://localhost:8501"
MODEL_NAME = "marketing-response-model"

## 1. Cek Status Model
Langkah pertama: pastikan model benar-benar termuat dan siap (`state: AVAILABLE`) sebelum mengirim request prediksi.

In [4]:
status_url = f"{BASE_URL}/v1/models/{MODEL_NAME}"
response = requests.get(status_url)
print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))

Status code: 200
{
  "model_version_status": [
    {
      "version": "1788165141",
      "state": "AVAILABLE",
      "status": {
        "error_code": "OK",
        "error_message": ""
      }
    }
  ]
}


## 2. Siapkan Data Contoh untuk Prediksi
Ambil beberapa baris dari data yang sudah dibersihkan (`marketing_campaign_clean.csv`)
sebagai contoh pelanggan yang akan diprediksi. Kolom `Response` (label asli)
disimpan terpisah hanya untuk pembanding — TIDAK dikirim ke model.

In [5]:
df = pd.read_csv('data/marketing_campaign_clean.csv')

# Ambil 3 baris contoh: 1 yang tahu labelnya Response=1, 1 yang Response=0, 1 acak
sample_df = pd.concat([
    df[df['Response'] == 1].head(1),
    df[df['Response'] == 0].head(1),
    df.sample(1, random_state=42),
]).reset_index(drop=True)

actual_labels = sample_df['Response'].tolist()
sample_df_for_request = sample_df.drop(columns=['Response'])
sample_df_for_request

,Education,Marital_Status,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Age,Customer_Tenure_Days,Total_Spending,Total_Children,Total_Purchases
0,Graduation,Single,58138.0,0,0,58,635,88,546,172,...,0,0,0,0,0,57,663,1617,0,25
1,Graduation,Single,46344.0,1,1,38,11,1,6,2,...,0,0,0,0,0,60,113,27,2,6
2,Graduation,Married,19107.0,1,0,49,2,4,9,10,...,0,0,0,0,0,34,311,46,1,6


## 3. Konversi ke Format `tf.Example` (Base64)
TF Serving REST API mengharapkan input dalam bentuk JSON dengan field
`examples` berisi list string ter-`base64` dari `tf.Example` yang sudah
di-serialize — ini karena signature model (`serve_tf_examples_fn` di
`marketing_trainer.py`) menerima input `serialized_tf_examples`.

In [6]:
def df_row_to_tf_example(row: pd.Series) -> tf.train.Example:
    feature = {}
    for col, val in row.items():
        if pd.api.types.is_float_dtype(type(val)) or isinstance(val, float):
            feature[col] = tf.train.Feature(float_list=tf.train.FloatList(value=[float(val)]))
        elif isinstance(val, (int,)):
            feature[col] = tf.train.Feature(int64_list=tf.train.Int64List(value=[int(val)]))
        else:
            feature[col] = tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(val).encode('utf-8')]))
    return tf.train.Example(features=tf.train.Features(feature=feature))


def build_request_payload(dataframe: pd.DataFrame) -> dict:
    examples_b64 = []
    for _, row in dataframe.iterrows():
        tf_example = df_row_to_tf_example(row)
        serialized = tf_example.SerializeToString()
        examples_b64.append({"b64": base64.b64encode(serialized).decode('utf-8')})
    return {"instances": examples_b64}


payload = build_request_payload(sample_df_for_request)
print("Jumlah instance yang dikirim:", len(payload['instances']))

Jumlah instance yang dikirim: 3


## 4. Kirim Prediction Request
Endpoint REST API standar TF Serving: `POST /v1/models/<model_name>:predict`.

In [10]:
requests.post(predict_url, data=json.dumps(payload))

<Response [422]>

In [12]:
predict_url = f"{BASE_URL}/v1/models/{MODEL_NAME}:predict"
response = requests.post(predict_url, data=json.dumps(payload))

print("Status code:", response.status_code)
result = response.json()
print(json.dumps(result, indent=2))

Status code: 200
{
  "predictions": [
    [
      0.7770309448242188
    ],
    [
      0.010324517264962196
    ],
    [
      0.019568372517824173
    ]
  ]
}


## 5. Interpretasi Hasil
Output model adalah probabilitas (`sigmoid`) bahwa pelanggan akan merespons
campaign (`Response = 1`). Bandingkan dengan label asli untuk sanity check.

In [13]:
import tensorflow as tf
model = tf.saved_model.load("serving_model/marketing-response-model/1788165141")
infer = model.signatures["serving_default"]
print(infer.structured_outputs)

{'output_0': TensorSpec(shape=(None, 1), dtype=tf.float32, name='output_0')}


In [14]:
predictions = result['predictions']

for i, (pred, actual) in enumerate(zip(predictions, actual_labels)):
    prob = pred[0] if isinstance(pred, list) else pred
    predicted_label = 1 if prob >= 0.5 else 0
    match = "✅ cocok" if predicted_label == actual else "❌ beda"
    print(f"Sample {i+1}: probabilitas Response=1 -> {prob:.4f} | prediksi: {predicted_label} | label asli: {actual} | {match}")

Sample 1: probabilitas Response=1 -> 0.7770 | prediksi: 1 | label asli: 1 | ✅ cocok
Sample 2: probabilitas Response=1 -> 0.0103 | prediksi: 0 | label asli: 0 | ✅ cocok
Sample 3: probabilitas Response=1 -> 0.0196 | prediksi: 0 | label asli: 0 | ✅ cocok


## Kesimpulan

Model yang di-serve lewat TensorFlow Serving berhasil diuji dengan mengirim
prediction request langsung ke REST API-nya (`/v1/models/{model}:predict`),
membuktikan bahwa:
1. Model dapat menerima request eksternal dalam format standar (`tf.Example` ter-base64).
2. Output probabilitas konsisten dengan hasil evaluasi model di notebook pipeline utama.
3. Sistem end-to-end (pipeline TFX → model serving → prediction request) berjalan utuh.